# Rewriting programmes beyond 3-Decomposition

This notebook gives a concrete first pass at applying the Collatz-style
rewrite/termination programme to **Barnette's Conjecture** and **Tutte's
5-Flow Conjecture**.

For Barnette, work in the dual. A Hamiltonian cycle in a cubic,
3-connected, bipartite plane graph corresponds to a partition of the
vertices of its Eulerian triangulation dual \(H\) into two induced trees.

For a nonempty induced subgraph \(F\), define

\[
\psi(F)=c(F)-1+\beta(F),\qquad
\beta(F)=|E(F)|-|V(F)|+c(F).
\]

For a binary colouring \(V(H)=A\sqcup B\), put

\[
\Phi(A,B)=\psi(H[A])+\psi(H[B]).
\]

Then \(\Phi=0\) exactly when both colour classes induce trees.

The rewrite moves tested here flip either one vertex or the two endpoints
of an edge, while never increasing \(\Phi\). The experiments are evidence,
not a proof of Barnette's Conjecture.

In [1]:
import sys, subprocess, importlib.util

for module, package in {
    "networkx": "networkx>=3.2",
    "pandas": "pandas>=2.0",
}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

from collections import defaultdict, deque
import itertools
import random
import networkx as nx
import pandas as pd

print("NetworkX:", nx.__version__)

NetworkX: 3.6.1


## 1. Plane dual and Barnette \(C_4\)-expansion

In [2]:
def planar_faces(G):
    planar, embedding = nx.check_planarity(G)
    if not planar:
        raise ValueError("Nonplanar graph.")
    seen, faces = set(), []
    for u, v in embedding.edges():
        if (u, v) not in seen:
            faces.append(embedding.traverse_face(u, v, seen))
    return faces


def planar_dual(G):
    faces = planar_faces(G)
    edge_faces = defaultdict(list)
    for index, face in enumerate(faces):
        for j, u in enumerate(face):
            v = face[(j + 1) % len(face)]
            edge_faces[tuple(sorted((u, v)))].append(index)

    assert all(len(items) == 2 for items in edge_faces.values())
    dual = nx.Graph()
    dual.add_nodes_from(range(len(faces)))
    for items in edge_faces.values():
        dual.add_edge(items[0], items[1])
    return dual, faces


def c4_expansion_candidates(G):
    candidates = []
    for face in planar_faces(G):
        length = len(face)
        for i in range(length):
            for j in range(i + 1, length):
                a, b = face[i], face[(i + 1) % length]
                c, d = face[j], face[(j + 1) % length]
                if len({a, b, c, d}) < 4:
                    continue

                first = max(G.nodes()) + 1
                p, q, r, s = range(first, first + 4)

                for crossed in (False, True):
                    H = G.copy()
                    H.remove_edge(a, b)
                    H.remove_edge(c, d)
                    H.add_edges_from([
                        (a, p), (p, q), (q, b),
                        (c, r), (r, s), (s, d),
                    ])
                    H.add_edges_from(
                        [(p, s), (q, r)] if crossed else [(p, r), (q, s)]
                    )

                    if set(dict(H.degree()).values()) != {3}:
                        continue
                    if not nx.is_bipartite(H):
                        continue
                    if not nx.check_planarity(H)[0]:
                        continue
                    candidates.append(H)
    return candidates


def generated_barnette_graph(steps, seed):
    rng = random.Random(seed)
    G = nx.cubical_graph()
    for _ in range(steps):
        choices = c4_expansion_candidates(G)
        if not choices:
            raise RuntimeError("No valid C4 expansion.")
        G = rng.choice(choices)

    assert nx.is_bipartite(G)
    assert nx.node_connectivity(G) >= 3
    assert set(dict(G.degree()).values()) == {3}
    return G

## 2. Fast exhaustive normalization test

Rather than constructing the full directed rewrite graph, we start from all
target colourings and traverse allowed moves backwards. A state is marked
successful exactly when it can reach a target without increasing \(\Phi\).

In [3]:
class BinaryColourRewrite:
    def __init__(self, H):
        self.H = nx.convert_node_labels_to_integers(H)
        self.n = self.H.number_of_nodes()
        self.full = (1 << self.n) - 1
        self.adj = [0] * self.n
        for u, v in self.H.edges():
            self.adj[u] |= 1 << v
            self.adj[v] |= 1 << u

    def components(self, mask):
        count = 0
        unseen = mask
        while unseen:
            count += 1
            seed = unseen & -unseen
            frontier = seed
            unseen ^= seed
            while frontier:
                reached = 0
                work = frontier
                while work:
                    bit = work & -work
                    work ^= bit
                    vertex = bit.bit_length() - 1
                    reached |= self.adj[vertex]
                frontier = reached & unseen
                unseen ^= frontier
        return count

    def side_defect(self, mask):
        if mask == 0:
            return self.n + 1

        vertices = mask.bit_count()
        twice_edges = 0
        work = mask
        while work:
            bit = work & -work
            work ^= bit
            vertex = bit.bit_length() - 1
            twice_edges += (self.adj[vertex] & mask).bit_count()

        edges = twice_edges // 2
        components = self.components(mask)
        cycle_rank = edges - vertices + components
        return (components - 1) + cycle_rank

    def all_values(self):
        return [
            self.side_defect(mask) + self.side_defect(self.full ^ mask)
            for mask in range(1 << self.n)
        ]

    def normalization_report(self, pair_flips=False):
        values = self.all_values()
        patches = [1 << v for v in range(self.n)]
        if pair_flips:
            patches += [(1 << u) | (1 << v) for u, v in self.H.edges()]

        targets = [mask for mask, value in enumerate(values) if value == 0]
        reached = bytearray(1 << self.n)
        queue = deque(targets)
        for mask in targets:
            reached[mask] = 1

        while queue:
            current = queue.popleft()
            current_value = values[current]
            for patch in patches:
                predecessor = current ^ patch
                # predecessor -> current is an allowed non-increasing move
                if not reached[predecessor] and values[predecessor] >= current_value:
                    reached[predecessor] = 1
                    queue.append(predecessor)

        return {
            "states": 1 << self.n,
            "targets": len(targets),
            "states_reaching_target": int(sum(reached)),
            "states_not_reaching_target": (1 << self.n) - int(sum(reached)),
        }

## 3. Exhaustive experiments on generated Barnette graphs

Single-vertex flipping eventually develops genuine traps. Adding adjacent
pair flips removes every trap in the tested complete colouring state spaces.

In [4]:
specifications = [
    (0, 1000),
    (1, 1100),
    (2, 1200),
    (3, 1300),
    (3, 1301),
    (4, 1400),
    (4, 1402),
    (5, 2000),
]

rows = []
for steps, seed in specifications:
    primal = generated_barnette_graph(steps, seed)
    dual, _ = planar_dual(primal)
    system = BinaryColourRewrite(dual)

    single = system.normalization_report(pair_flips=False)
    pair = system.normalization_report(pair_flips=True)

    rows.append({
        "C4_expansions": steps,
        "seed": seed,
        "primal_vertices": primal.number_of_nodes(),
        "dual_vertices": dual.number_of_nodes(),
        "states": single["states"],
        "targets": single["targets"],
        "single_flip_unresolved_states":
            single["states_not_reaching_target"],
        "pair_flip_unresolved_states":
            pair["states_not_reaching_target"],
    })

results = pd.DataFrame(rows)
results

,C4_expansions,seed,primal_vertices,dual_vertices,states,targets,single_flip_unresolved_states,pair_flip_unresolved_states
0,0,1000,8,6,64,12,0,0
1,1,1100,12,8,256,16,0,0
2,2,1200,16,10,1024,28,0,0
3,3,1300,20,12,4096,48,0,0
4,3,1301,20,12,4096,48,2,0
5,4,1400,24,14,16384,68,8,0
6,4,1402,24,14,16384,60,72,0
7,5,2000,28,16,65536,120,100,0


In [5]:
print("Generated Barnette graphs:", len(results))
print("Complete colouring states checked:", f"{results['states'].sum():,}")
print(
    "Graphs with single-flip traps:",
    int((results["single_flip_unresolved_states"] > 0).sum()),
)
print(
    "Unresolved states with adjacent-pair flips:",
    int(results["pair_flip_unresolved_states"].sum()),
)

assert results["pair_flip_unresolved_states"].sum() == 0

Generated Barnette graphs: 8
Complete colouring states checked: 107,840
Graphs with single-flip traps: 4
Unresolved states with adjacent-pair flips: 0


### Barnette Pair-Flip Conjecture

> For every Eulerian plane triangulation \(H\), every binary colouring can be
> transformed into a partition into two induced trees by repeatedly flipping
> either one vertex or two adjacent vertices, without increasing \(\Phi\).

This implies Barnette's Conjecture and is stronger because it asserts
normalization from every starting colouring.

The single-vertex version is false on the generated examples.

## 4. Exact Hamiltonian transfer table for a \(C_4\)-patch

In [6]:
terminals = ("a", "b", "c", "d")
internal = ("p", "q", "r", "s")
patch_edges = [
    ("a", "p"), ("p", "q"), ("q", "b"),
    ("c", "r"), ("r", "s"), ("s", "d"),
    ("p", "r"), ("q", "s"),
]

patterns = []
for mask in range(1 << len(patch_edges)):
    chosen = [patch_edges[i] for i in range(len(patch_edges)) if (mask >> i) & 1]
    degree = defaultdict(int)
    for u, v in chosen:
        degree[u] += 1
        degree[v] += 1

    if any(degree[v] != 2 for v in internal):
        continue
    if any(degree[v] not in (0, 1) for v in terminals):
        continue

    F = nx.Graph()
    F.add_nodes_from(terminals + internal)
    F.add_edges_from(chosen)

    relevant = [
        set(component)
        for component in nx.connected_components(F)
        if any(v in internal for v in component)
    ]
    if not all(any(degree[t] == 1 and t in component for t in terminals)
               for component in relevant):
        continue

    pairings = []
    for component in relevant:
        ends = [t for t in terminals if t in component and degree[t] == 1]
        if len(ends) == 2:
            pairings.append("-".join(sorted(ends)))

    patterns.append({
        "used_terminals": "".join(t for t in terminals if degree[t] == 1),
        "path_pairing": ", ".join(sorted(pairings)),
        "selected_edges": " ".join(f"{u}{v}" for u, v in chosen),
    })

pd.DataFrame(patterns)

,used_terminals,path_pairing,selected_edges
0,abcd,"a-b, c-d",ap pq qb cr rs sd
1,bd,b-d,pq qb rs sd pr
2,ac,a-c,ap pq cr rs qs
3,ab,a-b,ap qb rs pr qs
4,cd,c-d,pq cr sd pr qs
5,abcd,"a-c, b-d",ap qb cr sd pr qs


A Hamiltonian cycle of the parent graph that uses at least one of the two
expanded edges extends through the patch. The hard case is when all available
parent Hamiltonian cycles avoid both edges, so the rewrite certificate should
record a finite **face-interface signature**, not merely one Hamiltonian cycle.

## 5. Tutte 5-flow: a one-defect rewrite-or-cut lemma

Let \(f\) be an integer nowhere-zero \(6\)-flow with positive values in
\(\{1,2,3,4,5\}\), and suppose \(e=u\to v\) is the unique edge of value \(5\).

Delete \(e\) and form a residual digraph:

* value \(1\): traversable only with its orientation;
* value \(4\): traversable only against its orientation;
* values \(2,3\): traversable in both directions.

A residual \(u\)-to-\(v\) path gives a unit cycle correction reducing the
value \(5\) to \(4\), hence a nowhere-zero \(5\)-flow.

If no path exists, let \(S\) be the reachable set from \(u\). Every other
outgoing cut edge has value \(4\), and every incoming cut edge has value \(1\).
If their numbers are \(a\) and \(b\), flow conservation gives

\[
b=5+4a,\qquad |\delta(S)|=1+a+b=6+5a.
\]

Thus the unique defect is either removable by one cycle rewrite or certified
by a polarized cut of size \(6\bmod 5\).

## 6. Comparative verdict

* **3-Decomposition:** deepest framework so far; the unrestricted one-defect
  theorem remains plausible, but fixed-cycle augmentation and all bounded
  one/two-exchange repair claims fail.
* **Barnette:** strongest next target. The difficult \(C_4\)-expansion has a
  finite interface table, the naive single-flip system is falsified, and the
  adjacent-pair system survives every complete tested state space.
* **Tutte 5-flow:** elegant residual formulation, but much closer to standard
  flow augmentation and to an already active flow-reconfiguration literature.
* **Berge--Fulkerson:** natural multiplicity-defect rewriting exists, but the
  six-perfect-matching state space and alternating-cycle reconfiguration are
  substantially larger. It is a lower-priority first target.